In [8]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on 10.03.26
Nora Hirsch
"""

# Original script from Nora, adapted inbetween by Kathrin (manual groups), then automised and adapted by Nora

# ToDO: 
# try python file + write the bash script
# Implement tests and error messages for if stuff does not work.
# liveplotting?

import serial
# import time 
import csv
import time
from datetime import datetime
import os

import serial.tools.list_ports
import numpy as np

# Get a list of available serial ports
available_ports = serial.tools.list_ports.comports()


for port, desc, hwid in available_ports:
    print(f"Port: {port} | Description: {desc} | Hardware ID: {hwid}")

Port: /dev/cu.BLTH | Description: n/a | Hardware ID: n/a
Port: /dev/cu.usbserial-A922BJHF | Description: FT232R USB UART | Hardware ID: USB VID:PID=0403:6001 SER=A922BJHF LOCATION=20-4
Port: /dev/cu.usbserial-FT3GCNKB1 | Description: FT2232H device | Hardware ID: USB VID:PID=0403:6010 SER=FT3GCNKB LOCATION=20-2
Port: /dev/cu.usbserial-FT3GCNKB0 | Description: FT2232H device | Hardware ID: USB VID:PID=0403:6010 SER=FT3GCNKB LOCATION=20-2
Port: /dev/cu.Bluetooth-Incoming-Port | Description: n/a | Hardware ID: n/a
Port: /dev/cu.soundcoreSpaceQ45 | Description: n/a | Hardware ID: n/a


In [9]:
# read in config file, Experiment name and other commands:
# Get the absolute path of the script
script_dir = os.path.dirname(os.path.abspath("__file__"))

# Get the variables from the param txt file:
commands_lists = []
with open(os.path.join(script_dir, "param.txt"), "r") as file: #sys.argv[1]
        lines = file.readlines()
        for line in lines:
            if not line.startswith("#"): # Ignore lines starting with '#'
                commands = line.strip().split(';')
                commands_lists.append(commands)
Exp_name, Bath_freq, Bath_temps, MicroK_port, MicroK_Channels, MicroK_Zeropower, Logger_port, Logger_readout, Logger_groups, Logger_sensorNo = commands_lists

Nr_NTCs_group = int(Logger_groups[0])
Nr_MeasPoints = int(Logger_groups[1])
print("/dev/cu." + str(Logger_port[0]))

# print(MicroK_Channels)
# print(Logger_groupsize)
# print(Nr_NTCs_group)
# print(Nr_MeasPoints)
# print(Logger_readout)
# start time all the same for all the file names?:

#start_time = time.time()
start_time = datetime.now()

os.makedirs("Output", exist_ok=True)

/dev/cu.usbserial-FT3GCNKB0


In [10]:
# manage groups and commands:
# Nodes on and nodes off commands:
command_nodeson = "nodes on \r\n"
command_nodesoff = "nodes off \r\n"
command_wakeup = "help\r\n" #"????\
command_live = "LIVE \r\n"


# commands to switch the readouts on and off (e.g., GND, NTC1, TestSB):
Logger_positions = ['DateTime', 'TempADC', ' NTC1', ' NTC2', ' TestSB', ' TestN', ' GND', ' PRESSURE']
on_commands  = [f"{name.strip()} ON\r\n"  for name in Logger_positions if name.strip() in [a.strip() for a in Logger_readout]]
off_commands = [f"{name.strip()} OFF\r\n" for name in Logger_positions if name.strip() not in [a.strip() for a in Logger_readout]]
print(on_commands)
print(off_commands)


# seperate the NTCs into groups:
groups = [Logger_sensorNo[i:i+Nr_NTCs_group] for i in range(0, len(Logger_sensorNo), Nr_NTCs_group)]

# comand to switch the groups on:
commands_groupNTCs = [f"NODES {' '.join(g)} \r\n" for g in groups]
print(commands_groupNTCs)

# make the headers:
# Clean active list
active_clean = [a.strip() for a in Logger_readout]

# Standalone columns (no node prefix)
standalones = ["DateTime", "TempADC"]
standalone_cols = [s for s in standalones if s in active_clean]

# Per-node positions (excluding standalones)
node_positions = [a for a in active_clean if a not in standalones]

# Build node columns for each node
for group in groups:
    node_groups = []
    for node in group:
        cols = " | ".join([f"N{node}_{pos}" for pos in node_positions])
        node_groups.append(cols)

# make headers for the files:
headers = []
#file_names = []
#group_n = 1
for group in groups:
    node_groups = []
    #file_names.append(time.strftime("%Y%m%d-%H%M%S") + "_" + Exp_name[0] + "_Group" + str(group_n) + ".txt")
    #group_n += 1
    for node in group:
        cols = " | ".join([f"N{node.strip()}_{pos}" for pos in node_positions])
        node_groups.append(cols)
    header = "SecondsElapsed; DateTimePC; " + " || ".join(standalone_cols + node_groups)
    headers.append(header)


file_name = "Output/" + time.strftime("%Y%m%d-%H%M%S") + "_" + Exp_name[0] + "Logger.txt"
#for i, h in enumerate(headers):
#    print(f"Header {i+1}: {h}\n")

['NTC1 ON\r\n', 'NTC2 ON\r\n', 'TestSB ON\r\n']
['DateTime OFF\r\n', 'TempADC OFF\r\n', 'TestN OFF\r\n', 'GND OFF\r\n', 'PRESSURE OFF\r\n']
['NODES 29 91 93 98 99 \r\n']


In [ ]:
# Configure and open the serial port:
port =  "/dev/cu." + str(Logger_port[0]) #A9LMGQ4E "/dev/cu.usbserial-A922BJHF"#"/dev/cu.usbserial-FTG9EKPY"  # Update with your serial port
baudrate = 19200 #9600 #19200
bytesize = serial.EIGHTBITS  # 8 bits per byte
parity = serial.PARITY_NONE  # No parity
stopbits = serial.STOPBITS_ONE  # 1 stop bit
timeout = 1

# Open the serial port
ser = serial.Serial(port, baudrate, bytesize,parity, stopbits, timeout) #

if ser.isOpen():
    print("Serial port is open")
else:
    print("Failed to open serial port")


## wake head up:
ser.write(command_wakeup.encode("ascii"))

received_data = str()
while(not any(c.isalpha() for c in received_data)):
            
            #ser.write(command_header.encode("ascii"))
    received_data = ser.readline().decode("utf-8")
print(received_data)
print("Head has woken up.")
time.sleep(3)

ser.write(command_live.encode("ascii"))
time.sleep(3)
print("LIVE mode is being switched on.")

# test is LIVE mode is on. How?
# switch nodes on.(here also waking head up? making live mode?)
ser.write(command_nodeson.encode("ascii"))
time.sleep(2)
print("Nodes are being switched on.")
# Interacting with head automatically. Can I wake it up when it is asleep or do I always have to put "live" in the terminal?

time.sleep(3)
# active the relevant read-outs and deactivate the rest:
for command in on_commands + off_commands:
   print(f"Sending: {command.strip()}")
   ser.write(command.encode("ascii"))
   # ser.write(command.encode())  # uncomment for actual serial sending
   time.sleep(5)

In [ ]:
# simplify and save everything into one file"
#single_file_name = file_names[0]
start_time = datetime.now()

file_handle = open(file_name, "a")

try:
    while True:                                                         # loop forever until interrupted
        i = 0                                                           # reset measurement counter each full cycle
        for g_idx in range(len(headers)):
            ser.write(commands_groupNTCs[g_idx].encode("ascii"))       # send the NODES command for this group to serial
            r = False                                                   # reset flag: header not yet written for this group
            i_start = g_idx * (Nr_MeasPoints + 3)                     # first i value for this group's measurement window
            i_end   = (g_idx + 1) * (Nr_MeasPoints + 3)              # last i value for this group's measurement window
            while i_start <= i < i_end:                                # keep reading until this group has enough measurements
                received_data = str()                                  # reset received_data to empty string
                while not received_data:                               # keep trying until we actually get data
                    received_data = ser.readline().decode("utf-8")     # read one line from serial and decode it
                print(received_data)                                   # print to console for monitoring
                data_values = received_data.strip()                    # remove leading/trailing whitespace from data
                current_time = datetime.now()                          # get current timestamp
                elapsed_time = current_time - start_time              # calculate how long since start
                seconds_elapsed = elapsed_time.total_seconds()        # convert to seconds as a float
                current_datetime = str(current_time)                  # convert timestamp to string for writing
                if "New Node Array:" in received_data:                 # check if device confirmed the new node group
                    r = True                                           # set flag: now start writing data
                    file_handle.write(f"Group{g_idx+1}; " + headers[g_idx] + "\r\n")# write the header to this group's file
                if r:
                    file_handle.write(f"Group{g_idx+1}; {seconds_elapsed}; {current_datetime}; {data_values}\n")
                    file_handle.flush()
                    i += 1                                   # force write to disk immediately


except KeyboardInterrupt:
    print("Stopping...")
finally:
    file_handle.close()
    print("Files closed.")
    ser.close()
    print("Serial port closed")
        

In [ ]:
#save everything into 4 files
#from datetime import datetime
# Open all files dynamically
file_handles = [open(file_names[i], "a") for i in range(len(headers))]

try:
    while True:                                                         # loop forever until interrupted
        i = 0                                                           # reset measurement counter each full cycle
        for g_idx in range(len(headers)):
            ser.write(commands_groupNTCs[g_idx].encode("ascii"))       # send the NODES command for this group to serial
            r = False                                                   # reset flag: header not yet written for this group
            i_start = g_idx * (Nr_MeasPoints + 3)                     # first i value for this group's measurement window
            i_end   = (g_idx + 1) * (Nr_MeasPoints + 3)              # last i value for this group's measurement window
            while i_start <= i < i_end:                                # keep reading until this group has enough measurements
                received_data = str()                                  # reset received_data to empty string
                while not received_data:                               # keep trying until we actually get data
                    received_data = ser.readline().decode("utf-8")     # read one line from serial and decode it
                print(received_data)                                   # print to console for monitoring
                data_values = received_data.strip()                    # remove leading/trailing whitespace from data
                current_time = datetime.now()                          # get current timestamp
                elapsed_time = current_time - start_time              # calculate how long since start
                seconds_elapsed = elapsed_time.total_seconds()        # convert to seconds as a float
                current_datetime = str(current_time)                  # convert timestamp to string for writing
                if "New Node Array:" in received_data:                 # check if device confirmed the new node group
                    r = True                                           # set flag: now start writing data
                    file_handles[g_idx].write(headers[g_idx] + "\r\n")# write the header to this group's file
                if r:                                                  # only write data after header has been written
                    for f_idx, fh in enumerate(file_handles):         # loop over all open files
                        if f_idx == g_idx:                            # if this is the active group's file
                            fh.write(f"{seconds_elapsed}; {current_datetime}; {data_values}\n")  # write time + data
                        else:                                          # for all other groups' files
                            fh.write(f"{seconds_elapsed}; {current_datetime}\n")                 # write only timestamp (no data)
                        fh.flush()                                     # force write to disk immediately
                    i += 1                                             # increment counter only after valid data written


except KeyboardInterrupt:
    print("Stopping...")
finally:
    for fh in file_handles:
        fh.close()
    print("Files closed.")
    ser.close()
    print("Serial port closed")

In [ ]:
except KeyboardInterrupt:
    print("Stopping...")
finally:
    for fh in file_handles:
        fh.close()
    print("Files closed.")

In [ ]:
#time.sleep(5)
with open(csv_file1, "a") as data_file1:
 with open(csv_file2, "a") as data_file2:
  with open(csv_file3, "a") as data_file3:
   with open(csv_file4, "a") as data_file4:
    
    while(True):

        i = 0
        r = False
        #writer.writerow(header)
        #data_file.write(header1 + "\r\n")
        #print(header1)
        ser.write(command_nodesG1.encode("ascii"))
        #time.sleep(10)
        while i < (Nr_MeasPoints +3):
            # loooooop

            received_data = str()
        
            while(not received_data):
            
                #ser.write(command_header.encode("ascii"))
                received_data = ser.readline().decode("utf-8")
            
                # formatted_data = b'\n'.join([received_data[i:i+16] for i in range(0, len(received_data), 16)])
                #data_file.write(received_data)

            print(received_data)
            data_values = received_data.strip()
            #data_values = received_data.split(";")
            current_time = datetime.now() #.strftime("%Y-%m-%d %H:%M:%S.%f")
            elapsed_time = current_time - start_time
            seconds_elapsed = elapsed_time.total_seconds()
            current_datetime = str(current_time)
            
            if "New Node Array:" in received_data:
                r = True
                data_file1.write(header1 + "\r\n")
            if r:
                
                data_file1.write(f"{seconds_elapsed}; {current_datetime}; {data_values}\n")
                data_file1.flush()
                data_file2.write(f"{seconds_elapsed}; {current_datetime}\n")
                data_file2.flush()
                data_file3.write(f"{seconds_elapsed}; {current_datetime}\n")
                data_file3.flush()
                data_file4.write(f"{seconds_elapsed}; {current_datetime}\n")
                data_file4.flush()
                
                #data_file.write(header1 + "\r\n")
        
                i+=1
        
        #writer.writerow(header)
        #data_file.write(header2 + "\r\n")
        ser.write(command_nodesG2.encode("ascii"))
        #print(header2)
        #time.sleep(10)
        r = False
        while (Nr_MeasPoints+2) < i < (2*Nr_MeasPoints +6):
            #llloooooooppiiiiii
            
            received_data = str()
        
            while(not received_data):
            
                #ser.write(command_header.encode("ascii"))
                received_data = ser.readline().decode("utf-8")
            
                # formatted_data = b'\n'.join([received_data[i:i+16] for i in range(0, len(received_data), 16)])
                #data_file.write(received_data)

            print(received_data)
            data_values = received_data.strip()
            #data_values = received_data.split(";")
            current_time = datetime.now() #.strftime("%Y-%m-%d %H:%M:%S.%f")
            elapsed_time = current_time - start_time
            seconds_elapsed = elapsed_time.total_seconds()
            current_datetime = str(current_time)

            if "New Node Array:" in received_data:
                r = True
                data_file2.write(header2 + "\r\n")
            if r:
                
                data_file1.write(f"{seconds_elapsed}; {current_datetime}\n")
                data_file1.flush()
                data_file2.write(f"{seconds_elapsed}; {current_datetime}; {data_values}\n")
                data_file2.flush()
                data_file3.write(f"{seconds_elapsed}; {current_datetime}\n")
                data_file3.flush()
                data_file4.write(f"{seconds_elapsed}; {current_datetime}\n")
                data_file4.flush()
                
                #data_file.write(header2 + "\r\n")
        
                i+=1
    
        
        ser.write(command_nodesG3.encode("ascii"))
        #print(header2)
        #time.sleep(10)
        r = False
        while (2*Nr_MeasPoints+5) < i < (3*Nr_MeasPoints + 9):
            #llloooooooppiiiiiilooop
            
            received_data = str()
        
            while(not received_data):
            
                #ser.write(command_header.encode("ascii"))
                received_data = ser.readline().decode("utf-8")
            
                # formatted_data = b'\n'.join([received_data[i:i+16] for i in range(0, len(received_data), 16)])
                #data_file.write(received_data)

            print(received_data)
            data_values = received_data.strip()
            #data_values = received_data.split(";")
            current_time = datetime.now() #.strftime("%Y-%m-%d %H:%M:%S.%f")
            elapsed_time = current_time - start_time
            seconds_elapsed = elapsed_time.total_seconds()
            current_datetime = str(current_time)

            if "New Node Array:" in received_data:
                r = True
                data_file3.write(header3 + "\r\n")
            if r:
                
                data_file1.write(f"{seconds_elapsed}; {current_datetime}\n")
                data_file1.flush()
                data_file2.write(f"{seconds_elapsed}; {current_datetime}\n")
                data_file2.flush()
                data_file3.write(f"{seconds_elapsed}; {current_datetime}; {data_values}\n")
                data_file3.flush()
                data_file4.write(f"{seconds_elapsed}; {current_datetime}\n")
                data_file4.flush()
                
                #data_file.write(header3 + "\r\n")
        
                i+=1
        
        ser.write(command_nodesG4.encode("ascii"))
        #print(header2)
        #time.sleep(10)
        r = False
        while (3*Nr_MeasPoints+8) < i < (4*Nr_MeasPoints +12):
            #llloooooooppiiiiiiloooooppiiiii
            
            received_data = str()
        
            while(not received_data):
            
                #ser.write(command_header.encode("ascii"))
                received_data = ser.readline().decode("utf-8")
            
                # formatted_data = b'\n'.join([received_data[i:i+16] for i in range(0, len(received_data), 16)])
                #data_file.write(received_data)

            print(received_data)
            data_values = received_data.strip()
            #data_values = received_data.split(";")
            current_time = datetime.now() #.strftime("%Y-%m-%d %H:%M:%S.%f")
            elapsed_time = current_time - start_time
            seconds_elapsed = elapsed_time.total_seconds()
            current_datetime = str(current_time)

            if "New Node Array:" in received_data:
                r = True
                data_file4.write(header4 + "\r\n")
            if r:
                
                data_file1.write(f"{seconds_elapsed}; {current_datetime}\n")
                data_file1.flush()
                data_file2.write(f"{seconds_elapsed}; {current_datetime}\n")
                data_file2.flush()
                data_file3.write(f"{seconds_elapsed}; {current_datetime}\n")
                data_file3.flush()
                data_file4.write(f"{seconds_elapsed}; {current_datetime}; {data_values}\n")
                data_file4.flush()
                #data_file.write(header4 + "\r\n")
        
                i+=1
        
        
        



In [ ]:
ser.close() #help ??????

In [ ]:
isinstance(Nr_MeasPoints, str)